# Alpha Research — flipperAgent Phase 3B

**Goal:** Discover orthogonal alpha sources beyond the existing MeanReversion/TrendFollowing/Momentum models.

**Data:** Binance Futures OHLCV (no DB needed, direct API).

**Strategies to test:**
1. SuperTrend Flip — ATR-based trend following (indicator exists)
2. VWAP Deviation — Volume-weighted mean reversion (indicator exists)
3. Structure Zone Detection — S/R zones + volume confirm + MTF alignment (Telegram-style)
4. Combined Regime-Gated Ensemble

**Approach:** Vectorized backtest → performance metrics → Optuna hyperparameter sweep on winners.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from binance.um_futures import UMFutures
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

# Use existing indicators directly (pure computation, no side effects)
from libs.features.indicators import (
    RSI, EMA, MACD, BollingerBands, ATR, Supertrend, VWAP
)

plt.style.use('dark_background')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"NumPy {np.__version__}, Pandas {pd.__version__}")
print("Indicators loaded: RSI, EMA, MACD, BollingerBands, ATR, Supertrend, VWAP")

## 1. Data Fetching

Fetch 6 months of OHLCV data directly from Binance Futures.
Multiple timeframes for MTF analysis.

In [ ]:
# ── Data Fetching ──────────────────────────────────────────────────────

_RAW_COLS = [
    "timestamp", "open", "high", "low", "close", "volume", "close_time",
    "quote_vol", "trades", "taker_buy_base", "taker_buy_quote", "ignore",
]
OHLCV_COLS = ["timestamp", "open", "high", "low", "close", "volume"]
_MAX_LIMIT = 1500


def fetch_ohlcv(
    symbol: str,
    timeframe: str,
    days: int = 180,
    until: int | None = None,
) -> pd.DataFrame:
    """Fetch historical OHLCV from Binance Futures with auto-pagination."""
    client = UMFutures()  # public endpoints, no key needed for klines
    now_ms = int(datetime.now(timezone.utc).timestamp() * 1000)
    since = now_ms - days * 86_400_000
    end = until or now_ms

    frames = []
    cursor = since
    while cursor < end:
        lines = client.klines(symbol, timeframe, startTime=cursor, endTime=end, limit=_MAX_LIMIT)
        if not lines:
            break
        df = pd.DataFrame(lines, columns=_RAW_COLS)[OHLCV_COLS]
        for c in OHLCV_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        frames.append(df)
        last_ts = int(df["timestamp"].iloc[-1])
        if last_ts <= cursor:
            break
        cursor = last_ts + 1
        if len(lines) < _MAX_LIMIT:
            break

    if not frames:
        return pd.DataFrame(columns=OHLCV_COLS)

    result = pd.concat(frames, ignore_index=True)
    result = result.drop_duplicates("timestamp").sort_values("timestamp").reset_index(drop=True)
    result["dt"] = pd.to_datetime(result["timestamp"], unit="ms", utc=True)
    result = result.set_index("dt")
    print(f"{symbol} {timeframe}: {len(result)} candles from {result.index[0]} to {result.index[-1]}")
    return result

In [ ]:
# ── Fetch data for primary pairs ──────────────────────────────────────
# BTC on 1h and 4h, ETH on 4h — matching pipeline config
# Also fetch 30m for comparison with Telegram signals

DAYS = 180  # 6 months

btc_1h = fetch_ohlcv("BTCUSDT", "1h", days=DAYS)
btc_4h = fetch_ohlcv("BTCUSDT", "4h", days=DAYS)
eth_4h = fetch_ohlcv("ETHUSDT", "4h", days=DAYS)

# Extra: 30m for higher-resolution signal testing
btc_30m = fetch_ohlcv("BTCUSDT", "30m", days=DAYS)

print(f"\nTotal candles fetched: {len(btc_1h) + len(btc_4h) + len(eth_4h) + len(btc_30m)}")

## 2. Feature Engineering

Compute all available indicators for each dataset using the existing library.

In [ ]:
# ── Feature computation using existing indicator library ──────────────

def compute_features(df: pd.DataFrame, include_vwap: bool = True) -> pd.DataFrame:
    """Compute all indicators and attach as columns."""
    out = df.copy()
    close = out["close"].values.tolist()
    high = out["high"].values.tolist()
    low = out["low"].values.tolist()
    hlc = list(zip(high, low, close))

    # RSI
    rsi = RSI(period=14)
    out["RSI"] = rsi.batch(close)

    # EMA fast/slow
    ema_fast = EMA(period=12)
    ema_slow = EMA(period=26)
    out["EMA_12"] = ema_fast.batch(close)
    out["EMA_26"] = ema_slow.batch(close)

    # MACD
    macd = MACD(fast_period=12, slow_period=26, signal_period=9)
    macd_vals = macd.batch(close)
    out["MACD_line"] = [v[0] if v else None for v in macd_vals]
    out["MACD_signal"] = [v[1] if v else None for v in macd_vals]
    out["MACD_hist"] = [v[2] if v else None for v in macd_vals]

    # Bollinger Bands
    bb = BollingerBands(period=20, num_std=2.0)
    bb_vals = bb.batch(close)
    out["BB_mid"] = [v[0] if v else None for v in bb_vals]
    out["BB_upper"] = [v[1] if v else None for v in bb_vals]
    out["BB_lower"] = [v[2] if v else None for v in bb_vals]
    # BB width (normalized volatility)
    out["BB_width"] = (out["BB_upper"] - out["BB_lower"]) / out["BB_mid"]

    # ATR
    atr = ATR(period=14)
    out["ATR"] = atr.batch(hlc)

    # SuperTrend
    st = Supertrend(period=10, multiplier=3.0)
    st_vals = st.batch(hlc)
    out["ST_value"] = [v[0] if v else None for v in st_vals]
    out["ST_dir"] = [v[1] if v else None for v in st_vals]

    # VWAP (needs volume and timestamp)
    if include_vwap and "volume" in out.columns:
        try:
            vwap = VWAP()
            ts_sec = (out["timestamp"] / 1000).values.tolist()
            vol = out["volume"].values.tolist()
            vwap_data = list(zip(high, low, close, vol, ts_sec))
            out["VWAP"] = vwap.batch(vwap_data)
            out["VWAP_dev"] = (out["close"] - out["VWAP"]) / out["VWAP"] * 100  # % deviation
        except Exception as e:
            print(f"VWAP computation failed: {e}")
            out["VWAP"] = None
            out["VWAP_dev"] = None

    # ── Derived features ──
    # Volume ratio (current / 20-period SMA)
    out["vol_sma20"] = out["volume"].rolling(20).mean()
    out["vol_ratio"] = out["volume"] / out["vol_sma20"]

    # Returns
    out["returns"] = out["close"].pct_change()
    out["log_returns"] = np.log(out["close"] / out["close"].shift(1))

    # Range as % of close
    out["range_pct"] = (out["high"] - out["low"]) / out["close"] * 100

    # Body ratio (body / full range) — measure of conviction
    body = abs(out["close"] - out["open"])
    full_range = out["high"] - out["low"]
    out["body_ratio"] = np.where(full_range > 0, body / full_range, 0)

    return out


# Compute features for all datasets
btc_1h_feat = compute_features(btc_1h)
btc_4h_feat = compute_features(btc_4h)
eth_4h_feat = compute_features(eth_4h)
btc_30m_feat = compute_features(btc_30m)

print(f"\nFeature columns ({len(btc_1h_feat.columns)}): {list(btc_1h_feat.columns)}")
btc_4h_feat.dropna().tail(3)

## 3. Vectorized Backtester

Simple but correct backtester with:
- Long/Short/Flat positions
- Fixed stop-loss and take-profit (single TP initially)
- Multi-target exit support (for Telegram-style signals)
- Commission modeling
- Performance metrics (Sharpe, MaxDD, Win Rate, Profit Factor)

In [ ]:
# ── Backtester ─────────────────────────────────────────────────────────

@dataclass
class BacktestResult:
    """Container for backtest output."""
    total_return: float
    sharpe: float
    max_drawdown: float
    win_rate: float
    profit_factor: float
    num_trades: int
    avg_trade_return: float
    equity_curve: pd.Series
    trades: pd.DataFrame

    def summary(self) -> str:
        return (
            f"Return: {self.total_return:+.2%} | Sharpe: {self.sharpe:.2f} | "
            f"MaxDD: {self.max_drawdown:.2%} | WinRate: {self.win_rate:.1%} | "
            f"PF: {self.profit_factor:.2f} | Trades: {self.num_trades} | "
            f"AvgTrade: {self.avg_trade_return:+.4%}"
        )


def vectorized_backtest(
    df: pd.DataFrame,
    signals: pd.Series,
    sl_pct: float = 0.02,
    tp_pct: float = 0.03,
    commission_pct: float = 0.0004,  # 4bps per side (taker)
    initial_capital: float = 10_000.0,
) -> BacktestResult:
    """
    Run a vectorized backtest with SL/TP.

    Parameters
    ----------
    df : DataFrame with 'close', 'high', 'low' columns
    signals : Series of {-1, 0, 1} aligned with df index
    sl_pct : Stop-loss as fraction of entry price
    tp_pct : Take-profit as fraction of entry price
    commission_pct : Commission per side as fraction
    initial_capital : Starting capital
    """
    close = df["close"].values
    high = df["high"].values
    low = df["low"].values
    sig = signals.values.astype(float)

    n = len(close)
    equity = np.full(n, initial_capital)
    trades_list = []

    position = 0  # 1=long, -1=short, 0=flat
    entry_price = 0.0
    entry_idx = 0
    capital = initial_capital

    for i in range(1, n):
        # Check SL/TP if in position
        if position != 0:
            if position == 1:
                sl_hit = low[i] <= entry_price * (1 - sl_pct)
                tp_hit = high[i] >= entry_price * (1 + tp_pct)
            else:  # short
                sl_hit = high[i] >= entry_price * (1 + sl_pct)
                tp_hit = low[i] <= entry_price * (1 - tp_pct)

            if sl_hit or tp_hit:
                if tp_hit:  # TP gets priority (assume hit first on same bar)
                    exit_price = entry_price * (1 + tp_pct * position)
                else:
                    exit_price = entry_price * (1 - sl_pct * position)

                pnl_pct = (exit_price / entry_price - 1) * position
                pnl_pct -= 2 * commission_pct  # entry + exit commission
                capital *= (1 + pnl_pct)

                trades_list.append({
                    "entry_idx": entry_idx, "exit_idx": i,
                    "entry_price": entry_price, "exit_price": exit_price,
                    "direction": position, "pnl_pct": pnl_pct,
                    "exit_type": "TP" if tp_hit else "SL",
                })
                position = 0

        # New signal
        if position == 0 and sig[i] != 0:
            position = int(sig[i])
            entry_price = close[i]
            entry_idx = i

        equity[i] = capital
        if position != 0:
            # Mark-to-market unrealized
            unrealized = (close[i] / entry_price - 1) * position
            equity[i] = capital * (1 + unrealized)

    # Close any remaining position at last close
    if position != 0:
        pnl_pct = (close[-1] / entry_price - 1) * position - 2 * commission_pct
        capital *= (1 + pnl_pct)
        trades_list.append({
            "entry_idx": entry_idx, "exit_idx": n - 1,
            "entry_price": entry_price, "exit_price": close[-1],
            "direction": position, "pnl_pct": pnl_pct,
            "exit_type": "EOD",
        })
        equity[-1] = capital

    # ── Metrics ──
    eq_series = pd.Series(equity, index=df.index)
    trades_df = pd.DataFrame(trades_list) if trades_list else pd.DataFrame(
        columns=["entry_idx", "exit_idx", "entry_price", "exit_price",
                 "direction", "pnl_pct", "exit_type"]
    )

    total_return = equity[-1] / initial_capital - 1
    returns = eq_series.pct_change().dropna()

    # Annualized Sharpe (assume hourly data ~8760 bars/year; adjust if needed)
    bars_per_year = 365 * 24  # will be adjusted per-TF
    sharpe = (returns.mean() / returns.std() * np.sqrt(bars_per_year)) if returns.std() > 0 else 0.0

    # Max drawdown
    peak = eq_series.cummax()
    dd = (eq_series - peak) / peak
    max_dd = dd.min()

    # Win rate & profit factor
    if len(trades_df) > 0:
        wins = trades_df[trades_df["pnl_pct"] > 0]
        losses = trades_df[trades_df["pnl_pct"] <= 0]
        win_rate = len(wins) / len(trades_df)
        gross_profit = wins["pnl_pct"].sum() if len(wins) > 0 else 0
        gross_loss = abs(losses["pnl_pct"].sum()) if len(losses) > 0 else 1e-9
        profit_factor = gross_profit / gross_loss
        avg_trade = trades_df["pnl_pct"].mean()
    else:
        win_rate = 0.0
        profit_factor = 0.0
        avg_trade = 0.0

    return BacktestResult(
        total_return=total_return,
        sharpe=sharpe,
        max_drawdown=max_dd,
        win_rate=win_rate,
        profit_factor=profit_factor,
        num_trades=len(trades_df),
        avg_trade_return=avg_trade,
        equity_curve=eq_series,
        trades=trades_df,
    )


def plot_equity(results: dict[str, BacktestResult], title: str = "Equity Curves"):
    """Plot equity curves for multiple strategies."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), height_ratios=[3, 1])

    for name, res in results.items():
        norm = res.equity_curve / res.equity_curve.iloc[0]  # normalize to 1.0
        ax1.plot(norm.index, norm.values, label=f"{name} ({res.total_return:+.1%})")

    ax1.set_title(title)
    ax1.legend()
    ax1.set_ylabel("Normalized Equity")
    ax1.grid(alpha=0.3)

    # Drawdowns for first strategy
    first = list(results.values())[0]
    peak = first.equity_curve.cummax()
    dd = (first.equity_curve - peak) / peak
    ax2.fill_between(dd.index, dd.values, 0, alpha=0.5, color='red')
    ax2.set_ylabel("Drawdown")
    ax2.set_xlabel("Date")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

print("Backtester ready.")

## 4. Strategy 1: SuperTrend Flip

**Hypothesis:** SuperTrend direction changes capture trend reversals with ATR-adaptive bands.
Structurally different from EMA crossover — uses HL2 ± ATR×multiplier instead of price-only EMAs.

**Signal logic:**
- Long when `ST_dir` flips from -1 to +1
- Short when `ST_dir` flips from +1 to -1
- Hold until opposite flip or SL/TP

In [ ]:
# ── Strategy 1: SuperTrend ─────────────────────────────────────────────

def strategy_supertrend(
    df: pd.DataFrame,
    period: int = 10,
    multiplier: float = 3.0,
    rsi_filter: bool = True,
    rsi_ob: int = 70,
    rsi_os: int = 30,
) -> pd.Series:
    """
    Generate signals on SuperTrend direction flips.
    Optional RSI filter: skip long if overbought, skip short if oversold.
    """
    # Recompute SuperTrend with custom params
    hlc = list(zip(
        df["high"].values.tolist(),
        df["low"].values.tolist(),
        df["close"].values.tolist(),
    ))
    st = Supertrend(period=period, multiplier=multiplier)
    st_vals = st.batch(hlc)
    st_dir = pd.Series(
        [v[1] if v else 0 for v in st_vals],
        index=df.index,
    )

    # Detect flips
    prev_dir = st_dir.shift(1)
    flip_long = (prev_dir == -1) & (st_dir == 1)
    flip_short = (prev_dir == 1) & (st_dir == -1)

    signals = pd.Series(0, index=df.index)
    signals[flip_long] = 1
    signals[flip_short] = -1

    # RSI filter
    if rsi_filter and "RSI" in df.columns:
        rsi = df["RSI"]
        signals[(signals == 1) & (rsi > rsi_ob)] = 0   # don't long when overbought
        signals[(signals == -1) & (rsi < rsi_os)] = 0  # don't short when oversold

    return signals


# ── Run on BTC 4h ──
sig_st = strategy_supertrend(btc_4h_feat)
res_st = vectorized_backtest(btc_4h_feat.dropna(), sig_st.loc[btc_4h_feat.dropna().index], sl_pct=0.02, tp_pct=0.04)
print(f"SuperTrend BTC/4h: {res_st.summary()}")

# ── Run on BTC 1h ──
sig_st_1h = strategy_supertrend(btc_1h_feat)
res_st_1h = vectorized_backtest(btc_1h_feat.dropna(), sig_st_1h.loc[btc_1h_feat.dropna().index], sl_pct=0.015, tp_pct=0.03)
print(f"SuperTrend BTC/1h: {res_st_1h.summary()}")

# ── Run on ETH 4h ──
sig_st_eth = strategy_supertrend(eth_4h_feat)
res_st_eth = vectorized_backtest(eth_4h_feat.dropna(), sig_st_eth.loc[eth_4h_feat.dropna().index], sl_pct=0.025, tp_pct=0.05)
print(f"SuperTrend ETH/4h: {res_st_eth.summary()}")

## 5. Strategy 2: VWAP Deviation Mean Reversion

**Hypothesis:** When price deviates significantly from VWAP, it tends to revert.
This is structurally different from BB mean-reversion (price anchor = volume-weighted, not SMA).

**Signal logic:**
- Long when `VWAP_dev < -threshold` (price far below VWAP)
- Short when `VWAP_dev > +threshold` (price far above VWAP)
- Volume filter: only trade when `vol_ratio > vol_min` (confirms activity)

In [ ]:
# ── Strategy 2: VWAP Deviation ────────────────────────────────────────

def strategy_vwap_deviation(
    df: pd.DataFrame,
    dev_threshold: float = 0.5,   # % deviation from VWAP
    vol_min: float = 1.2,         # minimum volume ratio
    rsi_confirm: bool = True,
    rsi_os: int = 40,
    rsi_ob: int = 60,
) -> pd.Series:
    """Mean-reversion signals based on VWAP deviation."""
    signals = pd.Series(0, index=df.index)

    if "VWAP_dev" not in df.columns or df["VWAP_dev"].isna().all():
        print("VWAP_dev not available")
        return signals

    vwap_dev = df["VWAP_dev"]
    vol_ratio = df.get("vol_ratio", pd.Series(2.0, index=df.index))

    long_cond = (vwap_dev < -dev_threshold) & (vol_ratio > vol_min)
    short_cond = (vwap_dev > dev_threshold) & (vol_ratio > vol_min)

    if rsi_confirm and "RSI" in df.columns:
        rsi = df["RSI"]
        long_cond = long_cond & (rsi < rsi_os)
        short_cond = short_cond & (rsi > rsi_ob)

    signals[long_cond] = 1
    signals[short_cond] = -1

    return signals


# ── Run on BTC 1h (best TF for VWAP intraday) ──
sig_vwap = strategy_vwap_deviation(btc_1h_feat.dropna())
res_vwap = vectorized_backtest(btc_1h_feat.dropna(), sig_vwap, sl_pct=0.015, tp_pct=0.025)
print(f"VWAP MR BTC/1h: {res_vwap.summary()}")

# Try more aggressive thresholds
sig_vwap_agg = strategy_vwap_deviation(btc_1h_feat.dropna(), dev_threshold=0.3, vol_min=1.0)
res_vwap_agg = vectorized_backtest(btc_1h_feat.dropna(), sig_vwap_agg, sl_pct=0.01, tp_pct=0.02)
print(f"VWAP MR (aggressive) BTC/1h: {res_vwap_agg.summary()}")

# BTC 30m
sig_vwap_30m = strategy_vwap_deviation(btc_30m_feat.dropna(), dev_threshold=0.3, vol_min=1.0)
res_vwap_30m = vectorized_backtest(btc_30m_feat.dropna(), sig_vwap_30m, sl_pct=0.01, tp_pct=0.02)
print(f"VWAP MR BTC/30m: {res_vwap_30m.summary()}")

## 6. Strategy 3: Structure Zone Detection (Telegram-Style)

Inspired by the Telegram signal format — identify S/R zones, confirm with volume + MTF alignment.

**Zone detection:**
- Swing highs/lows from rolling windows → cluster into zones
- Volume profile: where the most trading happened → support/resistance anchors

**Entry logic:**
- Price enters a support zone + volume confirmation + higher-TF trend alignment → Long
- Price enters a resistance zone + volume confirmation + higher-TF trend alignment → Short

**Exit:** Multi-target (4 TPs with decreasing allocation) + stop-loss below zone

In [ ]:
# ── Strategy 3: Structure Zones ───────────────────────────────────────

def find_swing_points(
    high: np.ndarray,
    low: np.ndarray,
    lookback: int = 5,
) -> tuple[list[int], list[int]]:
    """Find swing high and swing low indices."""
    swing_highs = []
    swing_lows = []
    n = len(high)
    for i in range(lookback, n - lookback):
        if high[i] == max(high[i - lookback:i + lookback + 1]):
            swing_highs.append(i)
        if low[i] == min(low[i - lookback:i + lookback + 1]):
            swing_lows.append(i)
    return swing_highs, swing_lows


def cluster_levels(
    prices: list[float],
    tolerance_pct: float = 0.5,
) -> list[tuple[float, float, int]]:
    """Cluster nearby price levels into zones. Returns (zone_low, zone_high, touch_count)."""
    if not prices:
        return []
    sorted_prices = sorted(prices)
    zones = []
    cluster = [sorted_prices[0]]

    for p in sorted_prices[1:]:
        if (p - cluster[0]) / cluster[0] * 100 <= tolerance_pct:
            cluster.append(p)
        else:
            zones.append((min(cluster), max(cluster), len(cluster)))
            cluster = [p]
    zones.append((min(cluster), max(cluster), len(cluster)))

    # Filter: only zones with >= 2 touches
    return [(lo, hi, cnt) for lo, hi, cnt in zones if cnt >= 2]


def strategy_structure_zones(
    df: pd.DataFrame,
    htf_df: pd.DataFrame | None = None,
    swing_lookback: int = 5,
    zone_tolerance_pct: float = 0.5,
    zone_recency: int = 100,  # only use zones from last N bars
    vol_min: float = 1.2,
    require_htf_trend: bool = True,
) -> pd.Series:
    """Structure zone entry signals with optional HTF trend filter."""
    high = df["high"].values
    low = df["low"].values
    close = df["close"].values
    signals = pd.Series(0, index=df.index)

    # Pre-compute HTF trend (SuperTrend direction on higher TF)
    htf_trend = None
    if htf_df is not None and require_htf_trend and "ST_dir" in htf_df.columns:
        # Resample HTF direction to LTF index via forward fill
        htf_st = htf_df["ST_dir"].dropna()
        htf_trend = htf_st.reindex(df.index, method="ffill")

    for i in range(zone_recency + swing_lookback, len(df)):
        # Rolling zone detection over recent history
        start = max(0, i - zone_recency)
        sh, sl_ = find_swing_points(high[start:i], low[start:i], lookback=swing_lookback)

        # Support zones from swing lows
        support_prices = [low[start + j] for j in sl_]
        support_zones = cluster_levels(support_prices, zone_tolerance_pct)

        # Resistance zones from swing highs
        resist_prices = [high[start + j] for j in sh]
        resist_zones = cluster_levels(resist_prices, zone_tolerance_pct)

        current_close = close[i]
        vol_ok = df["vol_ratio"].iloc[i] > vol_min if "vol_ratio" in df.columns else True

        # Check if price is near a support zone → Long
        for zone_lo, zone_hi, touches in support_zones:
            if zone_lo <= current_close <= zone_hi * 1.002:  # within zone + small buffer
                if vol_ok:
                    # HTF filter
                    if htf_trend is not None:
                        htf_val = htf_trend.iloc[i] if i < len(htf_trend) else 0
                        if htf_val != 1:  # HTF not bullish
                            continue
                    signals.iloc[i] = 1
                    break

        # Check if price is near a resistance zone → Short
        if signals.iloc[i] == 0:
            for zone_lo, zone_hi, touches in resist_zones:
                if zone_lo * 0.998 <= current_close <= zone_hi:  # within zone
                    if vol_ok:
                        if htf_trend is not None:
                            htf_val = htf_trend.iloc[i] if i < len(htf_trend) else 0
                            if htf_val != -1:  # HTF not bearish
                                continue
                        signals.iloc[i] = -1
                        break

    return signals


# ── Run on BTC 1h with 4h as HTF ──
print("Running structure zone strategy (this may take a moment)...")
df_clean = btc_1h_feat.dropna()
sig_zones = strategy_structure_zones(
    df_clean,
    htf_df=btc_4h_feat,
    swing_lookback=5,
    zone_recency=100,
    vol_min=1.0,
)
res_zones = vectorized_backtest(df_clean, sig_zones, sl_pct=0.015, tp_pct=0.03)
print(f"Structure Zones BTC/1h+4h HTF: {res_zones.summary()}")

# Without HTF filter
sig_zones_nohtf = strategy_structure_zones(
    df_clean,
    htf_df=None,
    require_htf_trend=False,
    vol_min=1.2,
)
res_zones_nohtf = vectorized_backtest(df_clean, sig_zones_nohtf, sl_pct=0.015, tp_pct=0.03)
print(f"Structure Zones BTC/1h (no HTF): {res_zones_nohtf.summary()}")

## 7. Strategy 4: ADX Regime-Gated Ensemble

**Hypothesis:** The existing models are correlated. An ADX-based regime filter can gate which model fires:
- High ADX (trending) → SuperTrend signals
- Low ADX (ranging) → VWAP mean-reversion signals

**ADX implementation** (custom, not in indicator library yet):

In [ ]:
# ── ADX indicator (custom implementation) ─────────────────────────────
from numba import njit

@njit(cache=True)
def _compute_adx(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int) -> tuple:
    """Compute ADX, +DI, -DI."""
    n = len(high)
    plus_di = np.full(n, np.nan)
    minus_di = np.full(n, np.nan)
    adx = np.full(n, np.nan)

    if n < period * 2:
        return adx, plus_di, minus_di

    # True Range, +DM, -DM
    tr = np.zeros(n)
    plus_dm = np.zeros(n)
    minus_dm = np.zeros(n)

    for i in range(1, n):
        hl = high[i] - low[i]
        hc = abs(high[i] - close[i - 1])
        lc = abs(low[i] - close[i - 1])
        tr[i] = max(hl, hc, lc)

        up_move = high[i] - high[i - 1]
        down_move = low[i - 1] - low[i]

        if up_move > down_move and up_move > 0:
            plus_dm[i] = up_move
        if down_move > up_move and down_move > 0:
            minus_dm[i] = down_move

    # Smoothed TR, +DM, -DM (Wilder's smoothing)
    atr_s = np.sum(tr[1:period + 1])
    pdm_s = np.sum(plus_dm[1:period + 1])
    mdm_s = np.sum(minus_dm[1:period + 1])

    dx_vals = np.zeros(n)

    for i in range(period, n):
        if i == period:
            pass  # already summed
        else:
            atr_s = atr_s - (atr_s / period) + tr[i]
            pdm_s = pdm_s - (pdm_s / period) + plus_dm[i]
            mdm_s = mdm_s - (mdm_s / period) + minus_dm[i]

        if atr_s > 0:
            pdi = 100.0 * pdm_s / atr_s
            mdi = 100.0 * mdm_s / atr_s
        else:
            pdi = 0.0
            mdi = 0.0

        plus_di[i] = pdi
        minus_di[i] = mdi

        di_sum = pdi + mdi
        if di_sum > 0:
            dx_vals[i] = 100.0 * abs(pdi - mdi) / di_sum

    # Smooth DX into ADX
    adx_start = period * 2
    if adx_start < n:
        adx_val = np.mean(dx_vals[period:adx_start])
        adx[adx_start - 1] = adx_val
        for i in range(adx_start, n):
            adx_val = (adx_val * (period - 1) + dx_vals[i]) / period
            adx[i] = adx_val

    return adx, plus_di, minus_di


def add_adx(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    """Add ADX, +DI, -DI columns."""
    out = df.copy()
    adx, pdi, mdi = _compute_adx(
        df["high"].values, df["low"].values, df["close"].values, period
    )
    out["ADX"] = adx
    out["+DI"] = pdi
    out["-DI"] = mdi
    return out


# Add ADX to our datasets
btc_1h_feat = add_adx(btc_1h_feat)
btc_4h_feat = add_adx(btc_4h_feat)
eth_4h_feat = add_adx(eth_4h_feat)
btc_30m_feat = add_adx(btc_30m_feat)

print(f"ADX stats (BTC 4h): mean={btc_4h_feat['ADX'].mean():.1f}, "
      f"median={btc_4h_feat['ADX'].median():.1f}")

In [ ]:
# ── Strategy 4: Regime-Gated Ensemble ─────────────────────────────────

def strategy_regime_ensemble(
    df: pd.DataFrame,
    adx_threshold: float = 25.0,
    # SuperTrend params (trending regime)
    st_period: int = 10,
    st_multiplier: float = 3.0,
    # VWAP params (ranging regime)
    vwap_dev: float = 0.4,
    vwap_vol_min: float = 1.0,
) -> pd.Series:
    """Route to SuperTrend in trending regime, VWAP MR in ranging regime."""
    signals = pd.Series(0, index=df.index)

    adx = df["ADX"] if "ADX" in df.columns else pd.Series(np.nan, index=df.index)

    trending = adx > adx_threshold
    ranging = adx <= adx_threshold

    # Get component signals
    st_signals = strategy_supertrend(df, period=st_period, multiplier=st_multiplier, rsi_filter=True)
    vwap_signals = strategy_vwap_deviation(df, dev_threshold=vwap_dev, vol_min=vwap_vol_min)

    # Gate by regime
    signals[trending & (st_signals != 0)] = st_signals[trending & (st_signals != 0)]
    signals[ranging & (vwap_signals != 0)] = vwap_signals[ranging & (vwap_signals != 0)]

    return signals


# ── Run on BTC 1h ──
df_clean = btc_1h_feat.dropna()
sig_regime = strategy_regime_ensemble(df_clean)
res_regime = vectorized_backtest(df_clean, sig_regime, sl_pct=0.015, tp_pct=0.03)
print(f"Regime Ensemble BTC/1h: {res_regime.summary()}")

# Try different ADX thresholds
for adx_t in [20, 25, 30, 35]:
    sig = strategy_regime_ensemble(df_clean, adx_threshold=adx_t)
    res = vectorized_backtest(df_clean, sig, sl_pct=0.015, tp_pct=0.03)
    print(f"  ADX={adx_t}: {res.summary()}")

## 8. Comparison Dashboard

Side-by-side comparison of all strategies.

In [ ]:
# ── Compare all strategies ────────────────────────────────────────────

# Collect results (use BTC 1h as common benchmark where possible)
df_clean = btc_1h_feat.dropna()

all_results = {}

# SuperTrend
sig = strategy_supertrend(df_clean)
all_results["SuperTrend"] = vectorized_backtest(df_clean, sig, sl_pct=0.015, tp_pct=0.03)

# VWAP Mean Reversion
sig = strategy_vwap_deviation(df_clean, dev_threshold=0.4, vol_min=1.0)
all_results["VWAP MR"] = vectorized_backtest(df_clean, sig, sl_pct=0.015, tp_pct=0.025)

# Structure Zones
sig = strategy_structure_zones(df_clean, htf_df=btc_4h_feat, vol_min=1.0)
all_results["Struct Zones"] = vectorized_backtest(df_clean, sig, sl_pct=0.015, tp_pct=0.03)

# Regime Ensemble
sig = strategy_regime_ensemble(df_clean)
all_results["Regime Ensemble"] = vectorized_backtest(df_clean, sig, sl_pct=0.015, tp_pct=0.03)

# Buy & Hold baseline
bh_signals = pd.Series(0, index=df_clean.index)
bh_signals.iloc[0] = 1  # buy and hold
all_results["Buy & Hold"] = vectorized_backtest(df_clean, bh_signals, sl_pct=1.0, tp_pct=10.0)

# ── Summary table ──
summary_rows = []
for name, res in all_results.items():
    summary_rows.append({
        "Strategy": name,
        "Return": f"{res.total_return:+.2%}",
        "Sharpe": f"{res.sharpe:.2f}",
        "MaxDD": f"{res.max_drawdown:.2%}",
        "WinRate": f"{res.win_rate:.1%}",
        "ProfitFactor": f"{res.profit_factor:.2f}",
        "Trades": res.num_trades,
        "AvgTrade": f"{res.avg_trade_return:+.4%}",
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*90)
print("STRATEGY COMPARISON — BTC/USDT 1h (6 months)")
print("="*90)
print(summary_df.to_string(index=False))
print("="*90)

In [ ]:
# ── Equity curve visualization ────────────────────────────────────────
plot_equity(all_results, title="Strategy Comparison — BTC/USDT 1h (6 months)")

In [ ]:
# ── Signal distribution analysis ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (name, res) in zip(axes.flat, [(k, v) for k, v in all_results.items() if k != "Buy & Hold"]):
    if len(res.trades) > 0:
        ax.hist(res.trades["pnl_pct"] * 100, bins=30, alpha=0.7, edgecolor='white')
        ax.axvline(0, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f"{name} — Trade PnL Distribution")
        ax.set_xlabel("PnL (%)")
        ax.set_ylabel("Count")
    else:
        ax.text(0.5, 0.5, "No trades", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name)

plt.tight_layout()
plt.show()

## 9. Correlation Analysis

Check if the new strategies are truly orthogonal to each other and to the existing models.

In [ ]:
# ── Signal correlation ────────────────────────────────────────────────

# Compute signals for existing models too
df_clean = btc_1h_feat.dropna()

signal_matrix = pd.DataFrame(index=df_clean.index)

# Existing models (approximate their logic)
# MeanReversion: RSI oversold + below BB lower
rsi = df_clean["RSI"]
signal_matrix["MeanRev"] = np.where(
    (rsi <= 30) & (df_clean["close"] <= df_clean["BB_lower"]), 1,
    np.where((rsi >= 70) & (df_clean["close"] >= df_clean["BB_upper"]), -1, 0)
)

# TrendFollowing: EMA cross + MACD confirm
signal_matrix["TrendFollow"] = np.where(
    (df_clean["EMA_12"] > df_clean["EMA_26"]) & (df_clean["MACD_hist"] > 0), 1,
    np.where((df_clean["EMA_12"] < df_clean["EMA_26"]) & (df_clean["MACD_hist"] < 0), -1, 0)
)

# Momentum: RSI directional + MACD histogram
signal_matrix["Momentum"] = np.where(
    (rsi > 55) & (df_clean["MACD_hist"] > 0), 1,
    np.where((rsi < 45) & (df_clean["MACD_hist"] < 0), -1, 0)
)

# New strategies
signal_matrix["SuperTrend"] = strategy_supertrend(df_clean).values
signal_matrix["VWAP_MR"] = strategy_vwap_deviation(df_clean, dev_threshold=0.4).values
signal_matrix["Regime"] = strategy_regime_ensemble(df_clean).values

# Correlation matrix
corr = signal_matrix.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

# Annotate
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center',
                color='black' if abs(corr.iloc[i, j]) < 0.5 else 'white')

ax.set_title("Signal Correlation: Existing vs New Strategies")
plt.colorbar(im)
plt.tight_layout()
plt.show()

# Print correlation of new strategies vs existing
print("\nCorrelation of NEW strategies vs EXISTING models:")
existing = ["MeanRev", "TrendFollow", "Momentum"]
new = ["SuperTrend", "VWAP_MR", "Regime"]
print(corr.loc[new, existing].to_string())

## 10. Optuna Hyperparameter Optimization

Sweep hyperparameters on the most promising strategy.
Optimize for Sharpe ratio (risk-adjusted return).

In [ ]:
# ── Optuna sweep ──────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

df_clean = btc_1h_feat.dropna()

# Split: first 70% train, last 30% validation
split_idx = int(len(df_clean) * 0.7)
train_df = df_clean.iloc[:split_idx]
val_df = df_clean.iloc[split_idx:]
print(f"Train: {len(train_df)} bars, Val: {len(val_df)} bars")


def objective_supertrend(trial):
    period = trial.suggest_int("period", 5, 20)
    multiplier = trial.suggest_float("multiplier", 1.5, 5.0, step=0.1)
    rsi_ob = trial.suggest_int("rsi_ob", 60, 85)
    rsi_os = trial.suggest_int("rsi_os", 15, 40)
    sl_pct = trial.suggest_float("sl_pct", 0.005, 0.04, step=0.005)
    tp_pct = trial.suggest_float("tp_pct", 0.01, 0.06, step=0.005)

    sig = strategy_supertrend(train_df, period=period, multiplier=multiplier,
                              rsi_filter=True, rsi_ob=rsi_ob, rsi_os=rsi_os)
    res = vectorized_backtest(train_df, sig, sl_pct=sl_pct, tp_pct=tp_pct)

    # Multi-objective: maximize Sharpe, minimize max drawdown
    if res.num_trades < 10:
        return -10.0  # penalize low trade count
    return res.sharpe


def objective_regime(trial):
    adx_threshold = trial.suggest_float("adx_threshold", 15, 40, step=1)
    st_period = trial.suggest_int("st_period", 5, 20)
    st_multiplier = trial.suggest_float("st_multiplier", 1.5, 5.0, step=0.1)
    vwap_dev = trial.suggest_float("vwap_dev", 0.2, 1.0, step=0.05)
    sl_pct = trial.suggest_float("sl_pct", 0.005, 0.04, step=0.005)
    tp_pct = trial.suggest_float("tp_pct", 0.01, 0.06, step=0.005)

    sig = strategy_regime_ensemble(
        train_df, adx_threshold=adx_threshold,
        st_period=st_period, st_multiplier=st_multiplier,
        vwap_dev=vwap_dev,
    )
    res = vectorized_backtest(train_df, sig, sl_pct=sl_pct, tp_pct=tp_pct)

    if res.num_trades < 10:
        return -10.0
    return res.sharpe


# ── Run optimization ──
print("\nOptimizing SuperTrend...")
study_st = optuna.create_study(direction="maximize", study_name="SuperTrend")
study_st.optimize(objective_supertrend, n_trials=200, show_progress_bar=True)

print(f"\nBest SuperTrend Sharpe (train): {study_st.best_value:.2f}")
print(f"Best params: {study_st.best_params}")

print("\nOptimizing Regime Ensemble...")
study_regime = optuna.create_study(direction="maximize", study_name="RegimeEnsemble")
study_regime.optimize(objective_regime, n_trials=200, show_progress_bar=True)

print(f"\nBest Regime Ensemble Sharpe (train): {study_regime.best_value:.2f}")
print(f"Best params: {study_regime.best_params}")

In [ ]:
# ── Validate best params on held-out data ─────────────────────────────

print("=" * 70)
print("OUT-OF-SAMPLE VALIDATION (last 30%)")
print("=" * 70)

# SuperTrend
bp = study_st.best_params
sig_val = strategy_supertrend(
    val_df, period=bp["period"], multiplier=bp["multiplier"],
    rsi_ob=bp["rsi_ob"], rsi_os=bp["rsi_os"],
)
res_val_st = vectorized_backtest(val_df, sig_val, sl_pct=bp["sl_pct"], tp_pct=bp["tp_pct"])
print(f"SuperTrend (OOS): {res_val_st.summary()}")

# Regime
bp = study_regime.best_params
sig_val = strategy_regime_ensemble(
    val_df, adx_threshold=bp["adx_threshold"],
    st_period=bp["st_period"], st_multiplier=bp["st_multiplier"],
    vwap_dev=bp["vwap_dev"],
)
res_val_regime = vectorized_backtest(val_df, sig_val, sl_pct=bp["sl_pct"], tp_pct=bp["tp_pct"])
print(f"Regime Ensemble (OOS): {res_val_regime.summary()}")

# ── Compare in-sample vs out-of-sample ──
print("\n" + "=" * 70)
print("OVERFIT CHECK")
print("=" * 70)

bp = study_st.best_params
sig_is = strategy_supertrend(train_df, period=bp["period"], multiplier=bp["multiplier"],
                             rsi_ob=bp["rsi_ob"], rsi_os=bp["rsi_os"])
res_is_st = vectorized_backtest(train_df, sig_is, sl_pct=bp["sl_pct"], tp_pct=bp["tp_pct"])

bp = study_regime.best_params
sig_is = strategy_regime_ensemble(train_df, adx_threshold=bp["adx_threshold"],
                                  st_period=bp["st_period"], st_multiplier=bp["st_multiplier"],
                                  vwap_dev=bp["vwap_dev"])
res_is_regime = vectorized_backtest(train_df, sig_is, sl_pct=bp["sl_pct"], tp_pct=bp["tp_pct"])

print(f"SuperTrend — IS Sharpe: {res_is_st.sharpe:.2f} | OOS Sharpe: {res_val_st.sharpe:.2f} | Decay: {(res_is_st.sharpe - res_val_st.sharpe)/max(res_is_st.sharpe, 0.01):.0%}")
print(f"Regime     — IS Sharpe: {res_is_regime.sharpe:.2f} | OOS Sharpe: {res_val_regime.sharpe:.2f} | Decay: {(res_is_regime.sharpe - res_val_regime.sharpe)/max(res_is_regime.sharpe, 0.01):.0%}")

# Plot OOS equity curves
plot_equity(
    {"SuperTrend (OOS)": res_val_st, "Regime Ensemble (OOS)": res_val_regime},
    title="Out-of-Sample Validation — Optimized Strategies",
)

## 11. ADX Regime Distribution

Understand the regime landscape across our data.

In [ ]:
# ── Regime analysis ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, df) in zip(axes, [
    ("BTC 1h", btc_1h_feat), ("BTC 4h", btc_4h_feat), ("ETH 4h", eth_4h_feat)
]):
    adx = df["ADX"].dropna()
    ax.hist(adx, bins=40, alpha=0.7, edgecolor='white')
    ax.axvline(25, color='red', linestyle='--', label='ADX=25 threshold')
    trending_pct = (adx > 25).mean() * 100
    ax.set_title(f"{name}\nTrending: {trending_pct:.0f}% | Ranging: {100-trending_pct:.0f}%")
    ax.set_xlabel("ADX")
    ax.legend()

plt.suptitle("ADX Distribution — Regime Breakdown", y=1.02)
plt.tight_layout()
plt.show()

## 12. Next Steps

Based on results above, decide:

1. **Which strategies show positive alpha?** (Sharpe > 1.0 OOS, positive PF)
2. **Are they orthogonal?** (low correlation with existing models)
3. **Implementation priority:**
   - Build ADX indicator in the indicator registry
   - Implement winning strategy as a new Model in `src/libs/models/`
   - Wire into StrategyWorker via ModelRegistry
   - Run live paper trading through existing pipeline

### Quick parameter export for implementation

In [ ]:
# ── Export best params for implementation handoff ─────────────────────

import json

handoff = {
    "phase": "3B",
    "date": datetime.now(timezone.utc).isoformat(),
    "strategies": {},
}

# SuperTrend
bp = study_st.best_params
handoff["strategies"]["SuperTrend"] = {
    "params": bp,
    "train_sharpe": round(res_is_st.sharpe, 2),
    "oos_sharpe": round(res_val_st.sharpe, 2),
    "oos_return": round(res_val_st.total_return, 4),
    "oos_max_dd": round(res_val_st.max_drawdown, 4),
    "oos_win_rate": round(res_val_st.win_rate, 3),
    "oos_trades": res_val_st.num_trades,
}

# Regime Ensemble
bp = study_regime.best_params
handoff["strategies"]["RegimeEnsemble"] = {
    "params": bp,
    "train_sharpe": round(res_is_regime.sharpe, 2),
    "oos_sharpe": round(res_val_regime.sharpe, 2),
    "oos_return": round(res_val_regime.total_return, 4),
    "oos_max_dd": round(res_val_regime.max_drawdown, 4),
    "oos_win_rate": round(res_val_regime.win_rate, 3),
    "oos_trades": res_val_regime.num_trades,
}

print(json.dumps(handoff, indent=2))

# Save to file
with open("alpha_research_results.json", "w") as f:
    json.dump(handoff, f, indent=2)
print("\nResults saved to research/alpha_research_results.json")